<a href="https://colab.research.google.com/github/dr-koehler-ai/FHIR-Dashboard/blob/main/01_fhir_fundamentals.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [47]:
patient = {
    "resourceType": "Patient",
    "id": "patient-001",
    "name": [
        {
            "use": "official",
            "family": "Mueller",
            "given": ["Anna", "Maria"]
        }
    ],
    "gender": "female",
    "birthDate": "1959-03-12",
    "active": True
}

print(
    "Patient ID:", patient["id"],
    "\nFull Name:", patient["name"][0]["given"][0] + " " + patient["name"][0]["given"][1] + " " + patient["name"][0]["family"],
    "\nGender:", patient["gender"],
    "\nBirth Date:", patient["birthDate"],
    "\nActive:", patient["active"]
)

Patient ID: patient-001 
Full Name: Anna Maria Mueller 
Gender: female 
Birth Date: 1959-03-12 
Active: True


In [48]:
import pandas as pd

data = {"Patient ID": patient["id"],
    "Full Name": patient["name"][0]["given"][0] + " " + patient["name"][0]["given"][1] + " " + patient["name"][0]["family"],
    "Gender": patient["gender"],
    "Birth Date": patient["birthDate"],
    "Active": patient["active"]}

# 2. DataFrame erstellen mit einem expliziten Index
df = pd.DataFrame(data, index=[0])

# 3. DataFrame anzeigen
print(df)

    Patient ID           Full Name  Gender  Birth Date  Active
0  patient-001  Anna Maria Mueller  female  1959-03-12    True


In [49]:
patient_2 = {
    "resourceType": "Patient",
    "id": "patient-002",
    "name": [
        {
            "use": "maiden",
            "family": "Schmidt",
            "given": ["Lisa"]
        },
        {
            "use": "official",
            "family": "Weber",
            "given": ["Lisa", "Marie"]
        }
    ],
    "gender": "female",
    "birthDate": "1985-07-22",
    "active": True
}

def extract_patient_info(patient):

    # 1. Patient ID extrahieren
    patient_id = patient.get("id")

    # 2. Namen initialisieren
    full_name = None

    # 3. Zuerst nach dem offiziellen Namen suchen
    for name in patient.get("name", []):
        if name.get("use") == "official":
            first_names = name.get("given", [])
            last_name = name.get("family", "")

            full_name = " ".join(first_names + [last_name]).strip()
            break

    # 4. Falls kein offizieller Name vorhanden ist:
    #    Nach einem üblichen Namen suchen
    if full_name is None:
        for name in patient.get("name", []):
            if name.get("use") == "usual":
                first_names = name.get("given", [])
                last_name = name.get("family", "")

                full_name = " ".join(first_names + [last_name]).strip()
                break

    # 5. Weitere Patienteninformationen extrahieren
    gender = patient.get("gender")
    birth_date = patient.get("birthDate")
    status = patient.get("active")

    # 6. Vereinfachtes Dictionary zurückgeben
    return {
        "patient_id": patient_id,
        "full_name": full_name,
        "gender": gender,
        "birth_date": birth_date,
        "active": status
    }

print(extract_patient_info(patient))
print(extract_patient_info(patient_2))

{'patient_id': 'patient-001', 'full_name': 'Anna Maria Mueller', 'gender': 'female', 'birth_date': '1959-03-12', 'active': True}
{'patient_id': 'patient-002', 'full_name': 'Lisa Marie Weber', 'gender': 'female', 'birth_date': '1985-07-22', 'active': True}


In [50]:
patient_3 = {
    "resourceType": "Patient",
    "id": "patient-003",
    "name": [
        {
            "use": "usual",
            "family": "Fischer",
            "given": ["Max"]
        }
    ],
    "gender": "male"
}

print(extract_patient_info(patient))
print(extract_patient_info(patient_2))
print(extract_patient_info(patient_3))

{'patient_id': 'patient-001', 'full_name': 'Anna Maria Mueller', 'gender': 'female', 'birth_date': '1959-03-12', 'active': True}
{'patient_id': 'patient-002', 'full_name': 'Lisa Marie Weber', 'gender': 'female', 'birth_date': '1985-07-22', 'active': True}
{'patient_id': 'patient-003', 'full_name': 'Max Fischer', 'gender': 'male', 'birth_date': None, 'active': None}


In [51]:
patients = [patient, patient_2, patient_3]

patient_data = [
    extract_patient_info(p)
    for p in patients
]

df_patients = pd.DataFrame(patient_data)

df_patients

,patient_id,full_name,gender,birth_date,active
0,patient-001,Anna Maria Mueller,female,1959-03-12,True
1,patient-002,Lisa Marie Weber,female,1985-07-22,True
2,patient-003,Max Fischer,male,None,None


In [52]:
import sqlite3

# Temporäre SQLite-Datenbank erstellen
conn = sqlite3.connect(":memory:")

patients_df = pd.DataFrame({
    "patient_id": ["patient-001", "patient-002"],
    "name": ["Anna Mueller", "Max Weber"]
})

observations_df = pd.DataFrame({
    "observation_id": ["observation-001", "observation-002", "observation-003"],
    "patient_id": ["patient-001", "patient-001", "patient-002"],
    "measurement": ["Heart rate", "Creatinine", "Heart rate"],
    "value": [92, 1.4, 78],
    "unit": ["bpm", "mg/dL", "bpm"]
})

# DataFrames in SQLite speichern
patients_df.to_sql("patients", conn, index=False, if_exists="replace")
observations_df.to_sql("observations", conn, index=False, if_exists="replace")

3

In [53]:
{
    "observation_id": ...,
    "patient_id": ...,
    "value": ...,
    "unit": ...
}

{'observation_id': Ellipsis,
 'patient_id': Ellipsis,
 'value': Ellipsis,
 'unit': Ellipsis}

In [54]:
query = """
SELECT p.name, o.measurement, o.value, o.unit
FROM patients p
INNER JOIN observations o
    ON p.patient_id = o.patient_id
WHERE p.name = "Anna Mueller"
"""

result = pd.read_sql_query(query, conn)

display(result)

,name,measurement,value,unit
0,Anna Mueller,Creatinine,1.4,mg/dL
1,Anna Mueller,Heart rate,92.0,bpm


In [55]:
reference = "Patient/patient-001"
reference.split("/")[1]

'patient-001'

In [56]:
observation = {
    "resourceType": "Observation",
    "id": "observation-001",
    "subject": {
        "reference": "Patient/patient-001"
    },
    "valueQuantity": {
        "value": 92,
        "unit": "beats/minute"
    }
}

print({
    "observation_id": observation["id"],
    "patient_id": observation["subject"]["reference"].split("/")[1],
    "value": observation["valueQuantity"]["value"],
    "unit": observation["valueQuantity"]["unit"]
})

{'observation_id': 'observation-001', 'patient_id': 'patient-001', 'value': 92, 'unit': 'beats/minute'}


In [57]:
import json

with open("patient_file_1.json", "r", encoding="utf-8") as file:
    data = json.load(file)

print(type(data))

print(data.keys())

print(len(data["entry"]))

print(data.get("resourceType"))

print(data["entry"][0]["resource"]["resourceType"])

print(data["type"])

<class 'dict'>
dict_keys(['resourceType', 'type', 'entry'])
466
Bundle
Patient
transaction


In [58]:
def resource_counts(bundle):
  resource_counts = {}

  for entry in bundle["entry"]:
    resource = entry["resource"]
    resource_type = resource["resourceType"]

    # Wenn der Typ noch nicht vorhanden ist:
    if resource_type not in resource_counts:
      resource_counts[resource_type] = 1

    # Wenn er bereits vorhanden ist:
    else:
      resource_counts[resource_type] += 1

  return resource_counts

print(resource_counts(data))

def max_resource(counts):
  return max(counts, key=counts.get)

print(max_resource(resource_counts(data)))

def count_resources(counts):
  return sum(counts.values())

print(count_resources(resource_counts(data)))

{'Patient': 1, 'Encounter': 31, 'Condition': 25, 'DiagnosticReport': 54, 'DocumentReference': 31, 'Claim': 44, 'ExplanationOfBenefit': 44, 'MedicationRequest': 13, 'AllergyIntolerance': 1, 'CareTeam': 2, 'CarePlan': 2, 'Device': 2, 'SupplyDelivery': 12, 'Medication': 5, 'MedicationAdministration': 5, 'Observation': 126, 'Procedure': 58, 'Immunization': 8, 'ImagingStudy': 1, 'Provenance': 1}
Observation
466


In [59]:
def missing_entries(bundle):
  missing_valuequantity = 0
  missing_datetime = 0

  for entry in bundle["entry"]:
    resource = entry["resource"]
    resource_type = resource["resourceType"]

    if resource["resourceType"] == "Observation":
      if "valueQuantity" not in resource:
        missing_valuequantity += 1
      if "effectiveDateTime" not in resource:
        missing_datetime += 1

  return missing_valuequantity, missing_datetime

print(missing_entries(data))

(15, 0)


In [60]:
patient_resource = next(
    entry["resource"]
    for entry in data["entry"]
    if entry["resource"]["resourceType"] == "Patient"
)

df = pd.DataFrame(patient_resource.items())

print(df)

def extract_patient_info(patient):

    # 1. Patient ID extrahieren
    patient_id = patient.get("id")

    # 2. Namen initialisieren
    full_name = None

    # 3. Zuerst nach dem offiziellen Namen suchen
    for name in patient.get("name", []):
        if name.get("use") == "official":
            first_names = name.get("given", [])
            last_name = name.get("family", "")

            full_name = " ".join(first_names + [last_name]).strip()
            break

    # 4. Falls kein offizieller Name vorhanden ist:
    #    Nach einem üblichen Namen suchen
    if full_name is None:
        for name in patient.get("name", []):
            if name.get("use") == "usual":
                first_names = name.get("given", [])
                last_name = name.get("family", "")

                full_name = " ".join(first_names + [last_name]).strip()
                break

    # 5. Weitere Patienteninformationen extrahieren
    gender = patient.get("gender")
    birth_date = patient.get("birthDate")

    # 6. Vereinfachtes Dictionary zurückgeben
    return {
        "patient_id": patient_id,
        "full_name": full_name,
        "gender": gender,
        "birth_date": birth_date,
    }

patient_info = extract_patient_info(patient_resource)
df_patient = pd.DataFrame([patient_info])

                       0                                                  1
0           resourceType                                            Patient
1                     id               37549f60-b5a3-69cd-dea6-5a71c4bc23cf
2                   meta  {'profile': ['http://hl7.org/fhir/us/core/Stru...
3                   text  {'status': 'generated', 'div': '<div xmlns="ht...
4              extension  [{'url': 'http://hl7.org/fhir/us/core/Structur...
5             identifier  [{'system': 'https://github.com/synthetichealt...
6                   name  [{'use': 'official', 'family': 'Mosciski958', ...
7                telecom  [{'system': 'phone', 'value': '555-987-5663', ...
8                 gender                                             female
9              birthDate                                         1981-05-25
10               address  [{'extension': [{'url': 'http://hl7.org/fhir/S...
11         maritalStatus  {'coding': [{'system': 'http://terminology.hl7...
12  multiple

In [61]:
condition = {
    "resourceType": "Condition",
    "id": "condition-001",
    "subject": {
        "reference": "Patient/patient-001"
    },
    "code": {
        "coding": [
            {
                "system": "http://snomed.info/sct",
                "code": "44054006",
                "display": "Diabetes mellitus type 2"
            }
        ]
    },
    "clinicalStatus": {
        "coding": [
            {
                "code": "active"
            }
        ]
    }
}

def extracting_conditions(bundle):
    conditions = []

    # 1. Referenzen der Patienten auf ihre tatsächlichen IDs abbilden
    patient_lookup = {}

    for entry in bundle.get("entry", []):
        resource = entry.get("resource", {})

        if resource.get("resourceType") == "Patient":
            patient_id = resource.get("id")
            full_url = entry.get("fullUrl")

            if full_url and patient_id:
                patient_lookup[full_url] = patient_id

            if patient_id:
                patient_lookup[f"Patient/{patient_id}"] = patient_id

    # 2. Condition-Ressourcen extrahieren
    for entry in bundle.get("entry", []):
        resource = entry.get("resource", {})

        if resource.get("resourceType") != "Condition":
            continue

        condition_id = resource.get("id")

        # Patientenreferenz auflösen
        patient_ref = resource.get("subject", {}).get("reference")
        patient_id = patient_lookup.get(patient_ref)

        # Diagnosenkodierung
        codings = resource.get("code", {}).get("coding", [])
        coding = codings[0] if codings else {}

        condition_code = coding.get("code")
        condition_system = coding.get("system")
        condition_display = coding.get("display")

        # Klinischer Status
        status_codings = resource.get("clinicalStatus", {}).get("coding", [])
        clinical_status = (
            status_codings[0].get("code")
            if status_codings
            else None
        )

        conditions.append({
            "condition_id": condition_id,
            "patient_id": patient_id,
            "condition_code": condition_code,
            "condition_system": condition_system,
            "condition_display": condition_display,
            "clinical_status": clinical_status
        })

    return conditions


# 3. Auf deine echten Synthea-Daten anwenden
conditions = extracting_conditions(data)

df_conditions = pd.DataFrame(conditions)

print(df_conditions.head())
print(df_conditions.shape)
print(df_conditions["patient_id"].isna().sum())

                           condition_id                            patient_id  \
0  37549f60-b5a3-69cd-7947-2fca8a37b349  37549f60-b5a3-69cd-dea6-5a71c4bc23cf   
1  37549f60-b5a3-69cd-8859-e688ec2e9380  37549f60-b5a3-69cd-dea6-5a71c4bc23cf   
2  37549f60-b5a3-69cd-0fd7-a8d0614d927d  37549f60-b5a3-69cd-dea6-5a71c4bc23cf   
3  37549f60-b5a3-69cd-ff73-3fb485b997f6  37549f60-b5a3-69cd-dea6-5a71c4bc23cf   
4  37549f60-b5a3-69cd-46fc-55a2d74be407  37549f60-b5a3-69cd-dea6-5a71c4bc23cf   

  condition_code        condition_system  \
0      224299000  http://snomed.info/sct   
1      266934004  http://snomed.info/sct   
2      706893006  http://snomed.info/sct   
3      713458007  http://snomed.info/sct   
4      160903007  http://snomed.info/sct   

                            condition_display clinical_status  
0         Received higher education (finding)          active  
1                 Transport problem (finding)          active  
2  Victim of intimate partner abuse (finding)        res

In [62]:
observation = {
    "resourceType": "Observation",
    "id": "observation-001",
    "subject": {
        "reference": "Patient/patient-001"
    },
    "valueQuantity": {
        "value": 92,
        "unit": "beats/minute"
    }
}

def extracting_observations(bundle):
    observations = []

    # 1. Referenzen der Patienten auf ihre tatsächlichen IDs abbilden
    patient_lookup = {}

    for entry in bundle.get("entry", []):
        resource = entry.get("resource", {})

        if resource.get("resourceType") == "Patient":
            patient_id = resource.get("id")
            full_url = entry.get("fullUrl")

            if full_url and patient_id:
                patient_lookup[full_url] = patient_id

            if patient_id:
                patient_lookup[f"Patient/{patient_id}"] = patient_id

    # 2. Observation-Ressourcen extrahieren
    for entry in bundle.get("entry", []):
        resource = entry.get("resource", {})

        if resource.get("resourceType") != "Observation":
            continue

        observation_id = resource.get("id")

        # Patientenreferenz auflösen
        patient_ref = resource.get("subject", {}).get("reference")
        patient_id = patient_lookup.get(patient_ref)

        # Observationkodierung
        value_quantity = resource.get("valueQuantity") or {}

        observation_value = value_quantity.get("value")
        observation_unit = value_quantity.get("unit")

        codings = resource.get("code", {}).get("coding", [])
        coding = codings[0] if codings else {}
        observation_code = coding.get("code")
        observation_system = coding.get("system")
        observation_display = coding.get("display")
        observation_datetime = resource.get("effectiveDateTime")

        # Observation Status
        observation_status = resource.get("status")

        observations.append({
        "observation_id": observation_id,
        "patient_id": patient_id,
        "observation_code": observation_code,
        "observation_system": observation_system,
        "observation_display": observation_display,
        "observation_value": observation_value,
        "observation_unit": observation_unit,
        "observation_datetime": observation_datetime,
        "observation_status": observation_status
        })

    return observations


# 3. Auf deine echten Synthea-Daten anwenden
observations = extracting_observations(data)

df_observations = pd.DataFrame(observations)

print(df_observations.head())
print(df_observations.shape)
print(df_observations.isna().sum())

                         observation_id                            patient_id  \
0  37549f60-b5a3-69cd-9be7-59f59b340d58  37549f60-b5a3-69cd-dea6-5a71c4bc23cf   
1  37549f60-b5a3-69cd-ed91-dcc26e7c6338  37549f60-b5a3-69cd-dea6-5a71c4bc23cf   
2  37549f60-b5a3-69cd-61a9-688d2db7493a  37549f60-b5a3-69cd-dea6-5a71c4bc23cf   
3  37549f60-b5a3-69cd-b1b6-a2712280650e  37549f60-b5a3-69cd-dea6-5a71c4bc23cf   
4  37549f60-b5a3-69cd-6f97-8f1458050a72  37549f60-b5a3-69cd-dea6-5a71c4bc23cf   

  observation_code observation_system  \
0           4548-4   http://loinc.org   
1           8302-2   http://loinc.org   
2          72514-3   http://loinc.org   
3          29463-7   http://loinc.org   
4          39156-5   http://loinc.org   

                                 observation_display  observation_value  \
0           Hemoglobin A1c/Hemoglobin.total in Blood               5.85   
1                                        Body Height             169.80   
2  Pain severity - 0-10 verbal numeric ra

In [63]:
print(
    df_observations[
        ["observation_code", "observation_display", "observation_unit"]
    ].head(10)
)

print(
    df_observations["observation_display"].value_counts().head(10)
)

df_observations["observation_datetime"] = pd.to_datetime(
    df_observations["observation_datetime"],
    errors="coerce",
    utc=True
)

print(df_observations["observation_datetime"].dtype)

print(df_observations["observation_datetime"].isna().sum())

df_observations["observation_code"].value_counts()
df_observations[df_observations["observation_code"]== "20565-8"].sort_values("observation_datetime")

  observation_code                                observation_display  \
0           4548-4           Hemoglobin A1c/Hemoglobin.total in Blood   
1           8302-2                                        Body Height   
2          72514-3  Pain severity - 0-10 verbal numeric rating [Sc...   
3          29463-7                                        Body Weight   
4          39156-5                      Body mass index (BMI) [Ratio]   
5          85354-9    Blood pressure panel with all children optional   
6           8867-4                                         Heart rate   
7           9279-1                                   Respiratory rate   
8           2339-0                     Glucose [Mass/volume] in Blood   
9           6299-2               Urea nitrogen [Mass/volume] in Blood   

  observation_unit  
0                %  
1               cm  
2          {score}  
3               kg  
4            kg/m2  
5             None  
6             /min  
7             /min  
8      

,observation_id,patient_id,observation_code,observation_system,observation_display,observation_value,observation_unit,observation_datetime,observation_status
15,37549f60-b5a3-69cd-5c19-20c99bb8bc52,37549f60-b5a3-69cd-dea6-5a71c4bc23cf,20565-8,http://loinc.org,"Carbon dioxide, total [Moles/volume] in Blood",21.86,mmol/L,2016-12-05 22:20:00+00:00,final
36,37549f60-b5a3-69cd-c6a6-8b93ccf7eeba,37549f60-b5a3-69cd-dea6-5a71c4bc23cf,20565-8,http://loinc.org,"Carbon dioxide, total [Moles/volume] in Blood",24.16,mmol/L,2018-08-20 22:20:00+00:00,final
62,37549f60-b5a3-69cd-3c61-cf59ca6547c0,37549f60-b5a3-69cd-dea6-5a71c4bc23cf,20565-8,http://loinc.org,"Carbon dioxide, total [Moles/volume] in Blood",23.82,mmol/L,2021-05-31 22:20:00+00:00,final
94,37549f60-b5a3-69cd-7d9d-403abab1f46f,37549f60-b5a3-69cd-dea6-5a71c4bc23cf,20565-8,http://loinc.org,"Carbon dioxide, total [Moles/volume] in Blood",28.47,mmol/L,2023-06-05 22:20:00+00:00,final
119,37549f60-b5a3-69cd-f918-59edf292a73c,37549f60-b5a3-69cd-dea6-5a71c4bc23cf,20565-8,http://loinc.org,"Carbon dioxide, total [Moles/volume] in Blood",23.11,mmol/L,2025-06-09 22:20:00+00:00,final


In [64]:
def build_patient_lookup(bundle):
  patient_lookup = {}

  for entry in bundle.get("entry", []):
      resource = entry.get("resource", {})

      if resource.get("resourceType") == "Patient":
          patient_id = resource.get("id")
          full_url = entry.get("fullUrl")

          if full_url and patient_id:
              patient_lookup[full_url] = patient_id

          if patient_id:
              patient_lookup[f"Patient/{patient_id}"] = patient_id
  # Wir nehmen nur Patienten in unser Lookup auf, für die eine tatsächliche Patient.id vorhanden ist.
  return patient_lookup

def extracting_conditions(bundle, patient_lookup):
    conditions = []

    # Condition-Ressourcen extrahieren
    for entry in bundle.get("entry", []):
        resource = entry.get("resource", {})

        if resource.get("resourceType") != "Condition":
            continue

        condition_id = resource.get("id")

        # Patientenreferenz auflösen
        patient_ref = resource.get("subject", {}).get("reference")
        patient_id = patient_lookup.get(patient_ref)

        # Diagnosenkodierung
        codings = resource.get("code", {}).get("coding", [])
        coding = codings[0] if codings else {}

        condition_code = coding.get("code")
        condition_system = coding.get("system")
        condition_display = coding.get("display")

        # Klinischer Status
        status_codings = resource.get("clinicalStatus", {}).get("coding", [])
        clinical_status = (
            status_codings[0].get("code")
            if status_codings
            else None
        )

        conditions.append({
            "condition_id": condition_id,
            "patient_id": patient_id,
            "condition_code": condition_code,
            "condition_system": condition_system,
            "condition_display": condition_display,
            "clinical_status": clinical_status
        })

    return conditions

patient_lookup = build_patient_lookup(data)

conditions = extracting_conditions(data, patient_lookup)

df_conditions = pd.DataFrame(conditions)

print(df_conditions.shape)
print(df_conditions["patient_id"].isna().sum())

def extracting_observations(bundle, patient_lookup):
    observations = []

    #Observation-Ressourcen extrahieren
    for entry in bundle.get("entry", []):
        resource = entry.get("resource", {})

        if resource.get("resourceType") != "Observation":
            continue

        observation_id = resource.get("id")

        # Patientenreferenz auflösen
        patient_ref = resource.get("subject", {}).get("reference")
        patient_id = patient_lookup.get(patient_ref)

        # Observationkodierung
        value_quantity = resource.get("valueQuantity") or {}

        observation_value = value_quantity.get("value")
        observation_unit = value_quantity.get("unit")

        codings = resource.get("code", {}).get("coding", [])
        coding = codings[0] if codings else {}
        observation_code = coding.get("code")
        observation_system = coding.get("system")
        observation_display = coding.get("display")
        observation_datetime = resource.get("effectiveDateTime")

        # Observation Status
        observation_status = resource.get("status")

        observations.append({
        "observation_id": observation_id,
        "patient_id": patient_id,
        "observation_code": observation_code,
        "observation_system": observation_system,
        "observation_display": observation_display,
        "observation_value": observation_value,
        "observation_unit": observation_unit,
        "observation_datetime": observation_datetime,
        "observation_status": observation_status
        })

    return observations

observations = extracting_observations(data, patient_lookup)
df_observations = pd.DataFrame(observations)

print(df_observations.shape)
print(df_observations["patient_id"].isna().sum())

(25, 6)
0
(126, 9)
0


In [65]:
raw_observations = [
    entry["resource"]
    for entry in data["entry"]
    if entry["resource"]["resourceType"] == "Observation"
]

observations_with_components = [
    obs
    for obs in raw_observations
    if obs.get("component", [])
]

print("Observations with components:", len(observations_with_components))

first_component_observation = observations_with_components[0]

print("Observation code:")
print(first_component_observation.get("code"))

print("\nComponents:")
print(first_component_observation.get("component"))

def extract_components(resource):
    components = []

    for component in resource.get("component", []):
        component_coding = component.get("code", {}).get("coding", [])

        coding = component_coding[0] if component_coding else {}

        component_code = coding.get("code")
        component_display = coding.get("display")
        component_system = coding.get("system")

        value_quantity = component.get("valueQuantity") or {}

        component_value = value_quantity.get("value")
        component_unit = value_quantity.get("unit")

        components.append({
            "code": component_code,
            "system": component_system,
            "display": component_display,
            "value": component_value,
            "unit": component_unit
        })

    return components

components = extract_components(first_component_observation)

print(components)
print(len(components))

def extracting_observations(bundle, patient_lookup):
    observations = []

    #Observation-Ressourcen extrahieren
    for entry in bundle.get("entry", []):
        resource = entry.get("resource", {})

        if resource.get("resourceType") != "Observation":
            continue

        observation_id = resource.get("id")

        # Patientenreferenz auflösen
        patient_ref = resource.get("subject", {}).get("reference")
        patient_id = patient_lookup.get(patient_ref)

        # Observationkodierung
        value_quantity = resource.get("valueQuantity") or {}

        observation_value = value_quantity.get("value")
        observation_unit = value_quantity.get("unit")

        codings = resource.get("code", {}).get("coding", [])
        coding = codings[0] if codings else {}
        observation_code = coding.get("code")
        observation_system = coding.get("system")
        observation_display = coding.get("display")
        observation_datetime = resource.get("effectiveDateTime")

        # Observation Status
        observation_status = resource.get("status")

        # Mögliche Components
        if resource.get("component"):
          components = extract_components(resource)

          for component in components:
              observations.append({
                  "observation_id": observation_id,
                  "patient_id": patient_id,
                  "observation_code": component.get("code"),
                  "observation_system": component.get("system"),
                  "observation_display": component.get("display"),
                  "observation_value": component.get("value"),
                  "observation_unit": component.get("unit"),
                  "observation_datetime": observation_datetime,
                  "observation_status": observation_status
              })

        else:
          observations.append({
          "observation_id": observation_id,
          "patient_id": patient_id,
          "observation_code": observation_code,
          "observation_system": observation_system,
          "observation_display": observation_display,
          "observation_value": observation_value,
          "observation_unit": observation_unit,
          "observation_datetime": observation_datetime,
          "observation_status": observation_status
          })

    return observations

observations = extracting_observations(data, patient_lookup)

df_observations = pd.DataFrame(observations)

print("Shape:", df_observations.shape)

print("\nMissing values:")
print(df_observations.isna().sum())

component_counts = [
    len(obs.get("component", []))
    for obs in observations_with_components
]

print("Components pro Observation:", component_counts)
print("Gesamtzahl Components:", sum(component_counts))

Observations with components: 10
Observation code:
{'coding': [{'system': 'http://loinc.org', 'code': '85354-9', 'display': 'Blood pressure panel with all children optional'}], 'text': 'Blood pressure panel with all children optional'}

Components:
[{'code': {'coding': [{'system': 'http://loinc.org', 'code': '8462-4', 'display': 'Diastolic Blood Pressure'}], 'text': 'Diastolic Blood Pressure'}, 'valueQuantity': {'value': 70, 'unit': 'mm[Hg]', 'system': 'http://unitsofmeasure.org', 'code': 'mm[Hg]'}}, {'code': {'coding': [{'system': 'http://loinc.org', 'code': '8480-6', 'display': 'Systolic Blood Pressure'}], 'text': 'Systolic Blood Pressure'}, 'valueQuantity': {'value': 129, 'unit': 'mm[Hg]', 'system': 'http://unitsofmeasure.org', 'code': 'mm[Hg]'}}]
[{'code': '8462-4', 'system': 'http://loinc.org', 'display': 'Diastolic Blood Pressure', 'value': 70, 'unit': 'mm[Hg]'}, {'code': '8480-6', 'system': 'http://loinc.org', 'display': 'Systolic Blood Pressure', 'value': 129, 'unit': 'mm[Hg]'}

In [66]:
complex_observation = observations_with_components[1]

print(complex_observation.get("code"))

first_complex_component = complex_observation["component"][0]

print(first_complex_component)
print(first_complex_component.keys())

df_missing_values = df_observations[
    df_observations["observation_value"].isna()
]

print(df_missing_values["observation_display"].value_counts().head(10))

assert df_observations["observation_code"].notna().all()

{'coding': [{'system': 'http://loinc.org', 'code': '93025-5', 'display': "Protocol for Responding to and Assessing Patients' Assets, Risks, and Experiences [PRAPARE]"}], 'text': "Protocol for Responding to and Assessing Patients' Assets, Risks, and Experiences [PRAPARE]"}
{'code': {'coding': [{'system': 'http://loinc.org', 'code': '76501-6', 'display': 'Within the last year, have you been afraid of your partner or ex-partner'}], 'text': 'Within the last year, have you been afraid of your partner or ex-partner'}, 'valueCodeableConcept': {'coding': [{'system': 'http://loinc.org', 'code': 'LA32-8', 'display': 'No'}], 'text': 'No'}}
dict_keys(['code', 'valueCodeableConcept'])
observation_display
Tobacco smoking status                                                                                                                        5
Within the last year, have you been afraid of your partner or ex-partner                                                                      5
Do you feel

In [67]:
def get_value_type(component):

    possible_types = [
        "valueQuantity",
        "valueCodeableConcept",
        "valueString",
        "valueBoolean"
    ]

    for value_type in possible_types:

        if value_type in component:
            return value_type

    return None

import pandas as pd

def process_fhir_bundle(bundle):

    # 1. Patient Lookup erstellen
    patient_lookup = build_patient_lookup(bundle)

    # 2. Patient-Ressourcen aus dem Bundle extrahieren
    patient_resources = [
        entry["resource"]
        for entry in bundle.get("entry", [])
        if entry.get("resource", {}).get("resourceType") == "Patient"
    ]

    # 3. Patienteninformationen extrahieren
    patients = [
        extract_patient_info(patient)
        for patient in patient_resources
    ]

    # 4. Conditions extrahieren
    conditions = extracting_conditions(bundle, patient_lookup)

    # 5. Observations extrahieren
    observations = extracting_observations(bundle, patient_lookup)

    # 6. DataFrames erstellen
    df_patients = pd.DataFrame(patients)
    df_conditions = pd.DataFrame(conditions)
    df_observations = pd.DataFrame(observations)

    return df_patients, df_conditions, df_observations

df_patients, df_conditions, df_observations = process_fhir_bundle(data)

print("Patients:", df_patients.shape)
print("Conditions:", df_conditions.shape)
print("Observations:", df_observations.shape)

assert len(df_patients) == 1
assert len(df_conditions) == 25
assert len(df_observations) == 231

assert df_conditions["patient_id"].notna().all()
assert df_observations["patient_id"].notna().all()

assert df_observations["observation_code"].notna().all()

Patients: (1, 4)
Conditions: (25, 6)
Observations: (231, 9)


In [68]:
component = {
    "code": {
        "coding": [{
            "code": "76501-6",
            "display": "Afraid of partner or ex-partner"
        }]
    },
    "valueCodeableConcept": {
        "coding": [{
            "system": "http://loinc.org",
            "code": "LA32-8",
            "display": "No"
        }],
        "text": "No"
    }
}

def extract_categorical_value(component):
    value_codeable_concept = component.get("valueCodeableConcept") or {}

    coding_list = value_codeable_concept.get("coding", [])
    coding = coding_list[0] if coding_list else {}

    return {
        "code": coding.get("code"),
        "system": coding.get("system"),
        "display": coding.get("display")
    }

component = complex_observation["component"][0]

result = extract_categorical_value(component)

print(result)

{'code': 'LA32-8', 'system': 'http://loinc.org', 'display': 'No'}


In [69]:
def extract_component_value(component):

    value_type = get_value_type(component)

    if value_type == "valueQuantity":
        value_quantity = component.get("valueQuantity") or {}

        value = value_quantity.get("value")
        unit = value_quantity.get("unit")

    elif value_type == "valueCodeableConcept":
        categorical = extract_categorical_value(component)

        value = categorical.get("display")
        unit = None

    elif value_type == "valueString":
        value = component.get("valueString")
        unit = None

    elif value_type == "valueBoolean":
        value = component.get("valueBoolean")
        unit = None

    else:
        value = None
        unit = None

    return {
        "value_type": value_type,
        "value": value,
        "unit": unit
    }

    # Numerischer Blutdruckwert
bp_component = first_component_observation["component"][0]

# Kategoriale PRAPARE-Antwort
prapare_component = complex_observation["component"][0]

print(extract_component_value(bp_component))
print(extract_component_value(prapare_component))

{'value_type': 'valueQuantity', 'value': 70, 'unit': 'mm[Hg]'}
{'value_type': 'valueCodeableConcept', 'value': 'No', 'unit': None}


In [70]:
def extract_components(resource):
    components = []

    for component in resource.get("component", []):
        component_coding = component.get("code", {}).get("coding", [])

        coding = component_coding[0] if component_coding else {}

        component_code = coding.get("code")
        component_display = coding.get("display")
        component_system = coding.get("system")

        extracted_value = extract_component_value(component)
        component_unit = extracted_value.get("unit")
        component_value = extracted_value.get("value")
        component_value_type = extracted_value.get("value_type")

        components.append({
            "code": component_code,
            "system": component_system,
            "display": component_display,
            "value": component_value,
            "unit": component_unit,
            "value_type": component_value_type
        })

    return components

components = extract_components(complex_observation)

print("Anzahl Components:", len(components))
print("Erste Component:", components[0])

Anzahl Components: 21
Erste Component: {'code': '76501-6', 'system': 'http://loinc.org', 'display': 'Within the last year, have you been afraid of your partner or ex-partner', 'value': 'No', 'unit': None, 'value_type': 'valueCodeableConcept'}


In [71]:
def extracting_observations(bundle, patient_lookup):
    observations = []

    #Observation-Ressourcen extrahieren
    for entry in bundle.get("entry", []):
        resource = entry.get("resource", {})

        if resource.get("resourceType") != "Observation":
            continue

        observation_id = resource.get("id")

        # Patientenreferenz auflösen
        patient_ref = resource.get("subject", {}).get("reference")
        patient_id = patient_lookup.get(patient_ref)

        # Observationkodierung
        extracted_value = extract_component_value(resource)

        observation_value = extracted_value.get("value")
        observation_unit = extracted_value.get("unit")
        #observation_value_type = extracted_value.get("value_type")

        codings = resource.get("code", {}).get("coding", [])
        coding = codings[0] if codings else {}
        observation_code = coding.get("code")
        observation_system = coding.get("system")
        observation_display = coding.get("display")
        observation_datetime = resource.get("effectiveDateTime")

        # Observation Status
        observation_status = resource.get("status")

        # Mögliche Components
        if resource.get("component"):
          components = extract_components(resource)

          for component in components:
              observations.append({
                  "observation_id": observation_id,
                  "patient_id": patient_id,
                  "observation_code": component.get("code"),
                  "observation_system": component.get("system"),
                  "observation_display": component.get("display"),
                  "observation_value": component.get("value"),
                  "observation_unit": component.get("unit"),
                  "observation_datetime": observation_datetime,
                  "observation_status": observation_status,
                  "observation_value_type": component.get("value_type")
              })

        else:
          observations.append({
          "observation_id": observation_id,
          "patient_id": patient_id,
          "observation_code": observation_code,
          "observation_system": observation_system,
          "observation_display": observation_display,
          "observation_value": observation_value,
          "observation_unit": observation_unit,
          "observation_datetime": observation_datetime,
          "observation_status": observation_status,
          "observation_value_type": get_value_type(resource)
          })

    return observations

df_patients, df_conditions, df_observations = process_fhir_bundle(data)

print(df_observations.shape)
print(df_observations["observation_value"].dtype)
print(df_observations["observation_value_type"].value_counts(dropna=False))
print(df_observations["observation_value"].isna().sum())

df_none = df_observations[
    df_observations["observation_value_type"].isna()
]

print(df_none[
    ["observation_display", "observation_value", "observation_value_type"]
].head(10))

simple_observation = next(
    obs for obs in raw_observations
    if not obs.get("component")
)

print("FHIR-Felder:", simple_observation.keys())
print("Erkannter Werttyp:", get_value_type(simple_observation))

print("Werttyp im JSON vorhanden:",
      "valueQuantity" in simple_observation)

missing_values = df_observations[
    df_observations["observation_value"].isna()
]

print(
    missing_values[
        ["observation_display", "observation_value_type"]
    ]
)

df_patients, df_conditions, df_observations = process_fhir_bundle(data)

print(df_observations["observation_value"].isna().sum())

print(
    df_observations[
        df_observations["observation_display"].isin(
            ["Tobacco smoking status", "Address"]
        )
    ][
        ["observation_display", "observation_value",
         "observation_value_type"]
    ].head(6)
)

(231, 10)
object
observation_value_type
valueQuantity           131
valueCodeableConcept     95
valueString               5
Name: count, dtype: int64
0
Empty DataFrame
Columns: [observation_display, observation_value, observation_value_type]
Index: []
FHIR-Felder: dict_keys(['resourceType', 'id', 'meta', 'status', 'category', 'code', 'subject', 'encounter', 'effectiveDateTime', 'issued', 'valueQuantity'])
Erkannter Werttyp: valueQuantity
Werttyp im JSON vorhanden: True
Empty DataFrame
Columns: [observation_display, observation_value_type]
Index: []
0
        observation_display               observation_value  \
17   Tobacco smoking status  Never smoked tobacco (finding)   
30                  Address                 485 Heller Fork   
63   Tobacco smoking status  Never smoked tobacco (finding)   
76                  Address                 485 Heller Fork   
117  Tobacco smoking status  Never smoked tobacco (finding)   
130                 Address                 485 Heller Fork   

 

In [73]:
print(df_observations.dtypes)
print(df_observations["observation_datetime"].head())
print(type(df_observations["observation_datetime"].iloc[0]))

observation_id            object
patient_id                object
observation_code          object
observation_system        object
observation_display       object
observation_value         object
observation_unit          object
observation_datetime      object
observation_status        object
observation_value_type    object
dtype: object
0    2016-12-05T22:20:00+00:00
1    2016-12-05T22:20:00+00:00
2    2016-12-05T22:20:00+00:00
3    2016-12-05T22:20:00+00:00
4    2016-12-05T22:20:00+00:00
Name: observation_datetime, dtype: object
<class 'str'>


In [75]:
df_observations["observation_datetime"] = pd.to_datetime(
    df_observations["observation_datetime"],
    errors="coerce",
    utc=True
)

print(df_observations["observation_datetime"].dtype)

print(
    df_observations["observation_datetime"]
    .isna()
    .sum()
)

print(df_observations["observation_datetime"].head())

print(df_observations["observation_datetime"].dtype)
print(df_observations["observation_datetime"].isna().sum())

datetime64[ns, UTC]
0
0   2016-12-05 22:20:00+00:00
1   2016-12-05 22:20:00+00:00
2   2016-12-05 22:20:00+00:00
3   2016-12-05 22:20:00+00:00
4   2016-12-05 22:20:00+00:00
Name: observation_datetime, dtype: datetime64[ns, UTC]
datetime64[ns, UTC]
0


In [84]:
df_numeric = df_observations[df_observations["observation_value_type"]=="valueQuantity"].copy()

print(df_numeric.shape)
print(df_numeric["observation_value"].dtype)

df_numeric["observation_value"] = pd.to_numeric(
    df_numeric["observation_value"],
    errors="coerce"
)

print(df_numeric["observation_value"].dtype)
print(df_numeric["observation_value"].isna().sum())

df_observations["numeric_value"] = pd.to_numeric(
    df_observations["observation_value"].where(
        df_observations["observation_value_type"] == "valueQuantity"
    ),
    errors="coerce"
)

print(df_observations["numeric_value"].dtype)
print(df_observations["numeric_value"].notna().sum())

df_observations["categorical_value"] = (
    df_observations["observation_value"].where(
        df_observations["observation_value_type"].isin(
            ["valueCodeableConcept", "valueString"]
        )
    )
)

print(df_observations["categorical_value"].notna().sum())

print(
    df_observations[
        ["observation_display", "numeric_value", "categorical_value"]
    ].head(20)
)
print(type(df_observations.loc[18, "categorical_value"]))


(131, 12)
object
float64
0
float64
131
100
                                  observation_display  numeric_value  \
0            Hemoglobin A1c/Hemoglobin.total in Blood           5.85   
1                                         Body Height         169.80   
2   Pain severity - 0-10 verbal numeric rating [Sc...           1.00   
3                                         Body Weight          80.50   
4                       Body mass index (BMI) [Ratio]          27.93   
5                            Diastolic Blood Pressure          70.00   
6                             Systolic Blood Pressure         129.00   
7                                          Heart rate          89.00   
8                                    Respiratory rate          13.00   
9                      Glucose [Mass/volume] in Blood          65.63   
10               Urea nitrogen [Mass/volume] in Blood          19.64   
11                  Creatinine [Mass/volume] in Blood           1.54   
12                   

In [ ]:
assert len(df_observations) == ...

assert df_observations["numeric_value"].notna().sum() == ...

assert df_observations["categorical_value"].notna().sum() == ...